# Khởi tạo và optimizer: Xavier, He, SGD momentum, Adam, AdamW

Bạn tự viết hai cách khởi tạo và hai optimizer bằng numpy, đối chiếu **từng bước** với `torch.optim`, rồi
tái hiện hai bảng của phần lý thuyết: std activation qua 10 tầng với khởi tạo sai và đúng (mục 5), và
năm optimizer trên MNIST (mục 6).

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm, lớp. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật, train, vẽ; test bỏ qua chúng.
  Cell đầu tiên tải MNIST vào `data/raw/mnist` nếu chưa có.
- Optimizer nhận `params` là danh sách cặp `(tham số, gradient)`, giống `MLP.params()` của bài 01.
- Chấm: `uv run pytest tracks/ai-engineer/modules/01-nn-from-scratch/exercises/intermediate/02-init-and-optimizers -q`

In [ ]:
# explore: giới hạn luồng BLAS, tìm gốc repo, tải MNIST nếu chưa có (khoảng 11 MB, chỉ lần đầu)
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")   # đặt TRƯỚC khi import numpy; ma trận nhỏ mà 16 luồng thì chậm hơn ~5 lần
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure

MNIST = ensure("mnist")
print(MNIST, sorted(p.name for p in MNIST.iterdir()))

In [ ]:
import gzip
import time

import numpy as np

## 1. Xavier và He

Mục 5 lý thuyết suy ra: giữ thang đo khi forward qua một tầng ReLU cần $$\mathrm{Var}(w) = 2/n_{in}$$ (**He**); thoả
hiệp forward và backward cho activation gần tuyến tính quanh 0 là $$\mathrm{Var}(w) = \frac{2}{n_{in} + n_{out}}$$
(**Xavier**). Viết cả hai dạng phân phối chuẩn, trả ma trận shape `(fan_in, fan_out)`, lấy số ngẫu
nhiên **chỉ** từ `rng`.

In [ ]:
def xavier_init(fan_in: int, fan_out: int, rng: np.random.Generator) -> np.ndarray:
    """Ma trận (fan_in, fan_out) phân phối chuẩn, trung bình 0, phương sai 2 / (fan_in + fan_out)."""
    return rng.normal(0, np.sqrt(2 / (fan_in + fan_out)), (fan_in, fan_out))   # tham số thứ hai là STD


def he_init(fan_in: int, fan_out: int, rng: np.random.Generator) -> np.ndarray:
    """Ma trận (fan_in, fan_out) phân phối chuẩn, trung bình 0, phương sai 2 / fan_in."""
    return rng.normal(0, np.sqrt(2 / fan_in), (fan_in, fan_out))

In [ ]:
# explore: 10 tầng 512 neuron, input chuẩn hoá: std activation tầng 1/5/10 và std gradient về input (bảng mục 5)
def tanh_act(z):
    return np.tanh(z), 1 - np.tanh(z) ** 2

def relu_act(z):
    return np.maximum(z, 0), (z > 0).astype(float)

def run(name, act, init, layers=10, width=512):
    rng = np.random.default_rng(0)
    h = rng.normal(size=(1000, width))
    Ws, ds, stds = [], [], []
    for _ in range(layers):
        W = init(width, width, rng)
        h, d = act(h @ W)
        Ws.append(W); ds.append(d); stds.append(h.std())
    g = rng.normal(size=h.shape)                          # gradient giả từ loss đổ về
    for W, d in zip(reversed(Ws), reversed(ds)):
        g = (g * d) @ W.T
    sat = f" | |h| > 0.99: {np.mean(np.abs(h) > 0.99):.1%}" if act is tanh_act else ""
    print(f"{name:18s} std activation tầng 1/5/10: {stds[0]:.2e} {stds[4]:.2e} {stds[9]:.2e}"
          f" | std gradient về input: {g.std():.2e}{sat}")

run("tanh, N(0, 0.01)", tanh_act, lambda i, o, rng: rng.normal(0, 0.01, (i, o)))
run("tanh, N(0, 1)", tanh_act, lambda i, o, rng: rng.normal(0, 1.0, (i, o)))
run("tanh, Xavier", tanh_act, xavier_init)
run("ReLU, Xavier", relu_act, xavier_init)
run("ReLU, He", relu_act, he_init)

## 2. SGD momentum và Adam/AdamW

Công thức ở bảng mục 6 lý thuyết. Ký hiệu $$g$$ là gradient, $$\eta$$ là `lr`.

- `SGDMomentum(lr, momentum)`: kiểu **PyTorch** $$v \leftarrow \mu v + g,\ \theta \leftarrow \theta - \eta v$$ (không phải kiểu
  sách $$v \leftarrow \mu v + \eta g$$). `momentum=0` là SGD thường.
- `Adam(lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.0)`: $$m, v$$ khởi tạo 0, bias correction theo
  số bước $$t$$ (bắt đầu từ 1), $$\theta \leftarrow \theta - \eta\,\hat m/(\sqrt{\hat v} + \epsilon)$$. Với `weight_decay` $$\lambda > 0$$, trước
  cập nhật đó trừ thẳng $$\eta\lambda\theta$$ vào tham số (**AdamW**, decoupled), không cộng vào gradient.
- `step(params)` nhận danh sách cặp `(tham số, gradient)`, luôn cùng thứ tự, cập nhật tham số **tại chỗ**.
  State (velocity, $$m$$, $$v$$) là một mảng cho mỗi tham số, tạo ở lần gọi đầu.

Test chạy 20 bước trên một hàm bậc hai và so với `torch.optim.SGD(momentum=...)` và `torch.optim.AdamW`
cùng siêu tham số, `atol=1e-6`.

In [ ]:
class SGDMomentum:
    """Momentum kiểu PyTorch: v ← μ·v + g, p ← p − lr·v (momentum = 0 là SGD thường)."""

    def __init__(self, lr: float, momentum: float):
        self.lr, self.momentum = lr, momentum
        self.velocity: list[np.ndarray] | None = None     # một state mỗi tham số, tạo ở bước đầu

    def step(self, params: list[tuple[np.ndarray, np.ndarray]]) -> None:
        """params: [(tham số, gradient), ...], cùng thứ tự mỗi lần gọi. Cập nhật tham số tại chỗ."""
        if self.velocity is None:
            self.velocity = [np.zeros_like(p) for p, _ in params]
        for (p, g), v in zip(params, self.velocity):
            v *= self.momentum
            v += g
            p -= self.lr * v


class Adam:
    """Adam có bias correction; weight_decay > 0 là AdamW (decoupled: p ← p − lr·λ·p, tách khỏi gradient)."""

    def __init__(self, lr: float, betas: tuple[float, float] = (0.9, 0.999), eps: float = 1e-8,
                 weight_decay: float = 0.0):
        self.lr, self.betas, self.eps, self.weight_decay = lr, betas, eps, weight_decay
        self.t = 0
        self.m: list[np.ndarray] | None = None
        self.v: list[np.ndarray] | None = None

    def step(self, params: list[tuple[np.ndarray, np.ndarray]]) -> None:
        if self.m is None:
            self.m = [np.zeros_like(p) for p, _ in params]
            self.v = [np.zeros_like(p) for p, _ in params]
        self.t += 1
        b1, b2 = self.betas
        for (p, g), m, v in zip(params, self.m, self.v):
            if self.weight_decay:
                p -= self.lr * self.weight_decay * p      # decoupled: không đi qua m, v
            m *= b1
            m += (1 - b1) * g
            v *= b2
            v += (1 - b2) * g * g
            m_hat = m / (1 - b1 ** self.t)                # bias correction: m, v bắt đầu từ 0
            v_hat = v / (1 - b2 ** self.t)
            p -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

## 3. So trên MNIST

MLP 784-128-10 dưới đây là bản tham khảo, đúng hàm `forward_backward` ở mục 4 lý thuyết (float32; bài
Intermediate 01 yêu cầu bạn tự dựng lại nó thành lớp `MLP`), khởi tạo bằng
`he_init` của bạn. Mỗi cấu hình train 3 epoch, batch 64, cùng seed. Cả phần này chạy khoảng 5 giây; nếu
optimizer của bạn đúng, các con số trùng bảng mục 6 (máy khác hoặc số luồng BLAS khác có thể lệch ở chữ số
thứ tư, vì float32 cộng theo thứ tự khác).

In [ ]:
# explore: MLP 784-128-10 của mục 4 lý thuyết (forward_backward), He init của bạn, vòng train nhận một optimizer
# loss_grads là bản tham khảo lấy từ mục 4 lý thuyết; bài Intermediate 01 yêu cầu bạn tự dựng lại nó thành lớp MLP.
def load_idx(path):
    with gzip.open(path) as f:
        data = f.read()
    ndim = data[3]
    shape = np.frombuffer(data, ">u4", ndim, offset=4)
    return np.frombuffer(data, np.uint8, offset=4 + 4 * ndim).reshape(shape)

X = load_idx(MNIST / "train-images-idx3-ubyte.gz").reshape(-1, 784).astype(np.float32) / 255
y = load_idx(MNIST / "train-labels-idx1-ubyte.gz").astype(np.int64)
X_test = load_idx(MNIST / "t10k-images-idx3-ubyte.gz").reshape(-1, 784).astype(np.float32) / 255
y_test = load_idx(MNIST / "t10k-labels-idx1-ubyte.gz").astype(np.int64)


def init(seed):
    rng = np.random.default_rng(seed)
    return [he_init(784, 128, rng).astype(np.float32), np.zeros(128, np.float32),
            he_init(128, 10, rng).astype(np.float32), np.zeros(10, np.float32)]


def loss_grads(P, xb, yb):
    W1, b1, W2, b2 = P
    z1 = xb @ W1 + b1
    h = np.maximum(z1, 0)
    z = h @ W2 + b2
    z = z - z.max(axis=1, keepdims=True)
    log_p = z - np.log(np.exp(z).sum(axis=1, keepdims=True))
    dz = np.exp(log_p)
    dz[np.arange(len(yb)), yb] -= 1
    dz /= len(yb)
    dz1 = (dz @ W2.T) * (z1 > 0)
    return -log_p[np.arange(len(yb)), yb].mean(), [xb.T @ dz1, dz1.sum(0), h.T @ dz, dz.sum(0)]


def train(opt, epochs=3, batch_size=64, seed=0):
    """Trả (train loss trung bình theo mẫu từng epoch, test accuracy)."""
    P, rng, curve = init(seed), np.random.default_rng(seed), []
    for _ in range(epochs):
        idx, total = rng.permutation(len(X)), 0.0
        for i in range(0, len(X), batch_size):
            b = idx[i:i + batch_size]
            loss, G = loss_grads(P, X[b], y[b])
            opt.step(list(zip(P, G)))
            total += loss * len(b)
        curve.append(total / len(X))
    logits = np.maximum(X_test @ P[0] + P[1], 0) @ P[2] + P[3]
    return curve, float((logits.argmax(axis=1) == y_test).mean())

In [ ]:
# explore: so optimizer trên MNIST, 3 epoch, batch 64, cùng seed (bảng "So trên MNIST" mục 6)
import matplotlib.pyplot as plt

configs = [("SGD lr 0.01", SGDMomentum(0.01, 0.0)), ("SGD lr 0.1", SGDMomentum(0.1, 0.0)),
           ("SGD + momentum 0.9, lr 0.01", SGDMomentum(0.01, 0.9)), ("Adam lr 1e-3", Adam(1e-3)),
           ("Adam lr 1e-1", Adam(1e-1))]
for name, opt in configs:
    start = time.perf_counter()
    curve, acc = train(opt)
    print(f"{name:28s} train loss theo epoch {' / '.join(f'{c:.3f}' for c in curve)} | test acc {acc:.4f}"
          f" | {time.perf_counter() - start:.1f} s")
    plt.plot(range(1, 4), curve, marker="o", label=name)
plt.xlabel("epoch"); plt.ylabel("train loss"); plt.yscale("log"); plt.xticks([1, 2, 3]); plt.legend()
plt.show()

## 4. Đọc kết quả

Trước khi làm trắc nghiệm trong app, tự trả lời:

1. Vì sao "SGD + momentum 0.9, lr 0.01" cho đường cong gần trùng "SGD lr 0.1"?
2. Adam lr 1e-3 tốt nhất mà không cần tune; vậy vì sao Adam lr 1e-1 hỏng? "Adam không cần tune lr" sai ở đâu?
3. Ở bảng khởi tạo, "tanh, N(0, 1)" có std activation đẹp nhất. Vì sao đó vẫn là khởi tạo tệ?